# Optimización de Hiperparámetros VCP con Optuna

Busca la mejor configuración de ~12 parámetros del detector VCP usando TPE (Tree-structured Parzen Estimator).

**Prerequisitos:**
```bash
pip install optuna plotly mlflow
```

## 1. Setup e imports

In [2]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import optuna

optuna.logging.set_verbosity(optuna.logging.INFO)
warnings.filterwarnings("ignore", category=FutureWarning, module="mlflow")

project_root = Path.cwd().parent.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from autoresearch import (
    SwingCache,
    filter_tickers_by_start_date,
    find_common_period,
    get_ticker_info,
    load_universe,
    build_objective_function,
    create_study,
    run_backtest_for_params,
    compute_objective_score,
    study_to_dataframe,
    top_trials_summary,
    param_importance,
    reconstruct_pipeline_params,
    trades_to_dataframe,
    MLflowOptunaLogger,
    DEFAULT_RISK_PARAMS,
)

pd.set_option("display.float_format", "{:.4f}".format)
DATA_DIR = project_root / "data" / "csv"
print(f"Project root: {project_root}")
print(f"Data dir: {DATA_DIR}")
print("Imports OK")

Project root: /home/gdelarosa/proyectos/deteccion-vcp
Data dir: /home/gdelarosa/proyectos/deteccion-vcp/data/csv
Imports OK


## 2. Cargar y filtrar universo de tickers

In [3]:
ticker_info = get_ticker_info(DATA_DIR)
display(ticker_info)

,start_date,end_date,n_bars
ticker,,,
AAPL,2015-01-02,2026-04-08,2832
AMZN,2015-01-02,2026-04-08,2832
AVGO,2015-01-02,2026-04-08,2832
BRK.B,2015-01-02,2026-04-10,2834
COIN,2021-04-14,2026-04-10,1254
GLD,2015-01-02,2026-04-10,2834
GOOGL,2015-01-02,2026-04-08,2832
HOOD,2021-07-29,2026-04-10,1180
IWM,2015-01-02,2026-04-10,2834


In [4]:
opt_tickers, oos_tickers = filter_tickers_by_start_date(
    DATA_DIR, max_start_date="2015-12-31", exclude_tickers=["META"]
)
print(f"Optimization set ({len(opt_tickers)}): {opt_tickers}")
print(f"Out-of-sample set ({len(oos_tickers)}): {oos_tickers}")

Optimization set (18): ['AAPL', 'AMZN', 'AVGO', 'BRK.B', 'GLD', 'GOOGL', 'IWM', 'JPM', 'MSFT', 'NVDA', 'QLD', 'QQQ', 'SLV', 'SPY', 'SQQQ', 'TIL', 'TLT', 'TQQQ']
Out-of-sample set (4): ['COIN', 'HOOD', 'PLTR', 'SOFI']


In [5]:
universe = load_universe(opt_tickers, DATA_DIR)
common_start, common_end = find_common_period(universe)
print(f"Universe: {len(universe)} tickers")
print(f"Common period: {common_start.date()} to {common_end.date()}")
print(f"Bars per ticker: {[len(df) for df in universe.values()]}")

Universe: 18 tickers
Common period: 2015-05-27 to 2026-04-08
Bars per ticker: [2832, 2832, 2832, 2834, 2834, 2832, 2834, 2834, 2832, 2832, 2832, 2832, 2834, 2832, 2832, 1814, 2832, 2832]


## 3. Baseline: parámetros del experimento original

Corremos el pipeline con los parámetros conocidos (`volume_ratio_threshold=1.5`) como referencia.

In [6]:
from models.configs import ATRZigZagConfig

BASELINE_PARAMS = {
    "swing_config": ATRZigZagConfig(atr_length=14, atr_mult=2.0, use_close_only=False),
    "sequence_params": {
        "method": "tolerance",
        "min_contractions": 2,
        "max_contractions": 6,
        "lookback_bars": 126,
        "tolerance": 0.10,
        "max_depth_pct": 0.35,
        "min_total_reduction": 0.80,
        "max_gap_between_contractions_days": None,
    },
    "compression_params": {
        "method": "ratio",
        "atr_period": 14,
        "ratio_threshold": 0.85,
    },
    "volume_contraction_params": {
        "method": "ratio",
        "volume_column": "volume",
        "ratio_threshold": 0.85,
    },
    "breakout_params": {
        "volume_method": "ratio",
        "volume_ratio_threshold": 1.5,
        "volume_lookback_days": 50,
        "require_volume_confirmation": True,
    },
    "grouping_params": {"max_gap_days": 30},
    "risk_params": dict(DEFAULT_RISK_PARAMS),
}

In [7]:
baseline_result = run_backtest_for_params(universe, BASELINE_PARAMS)
baseline_metrics = baseline_result["metrics"]
baseline_score = compute_objective_score(baseline_metrics)

print("=== BASELINE ===")
print(f"  n_trades:       {baseline_metrics['n_trades']}")
print(f"  expectancy_r:   {baseline_metrics['expectancy_r']:+.4f}")
print(f"  win_rate:       {baseline_metrics['win_rate']:.1%}")
print(f"  profit_factor:  {baseline_metrics['profit_factor']:.2f}")
print(f"  score (Opt C):  {baseline_score:+.4f}")
print(f"\n  Trades per ticker:")
for ticker, n in sorted(baseline_metrics["trades_per_ticker"].items()):
    print(f"    {ticker}: {n}")

=== BASELINE ===
  n_trades:       78
  expectancy_r:   +0.3058
  win_rate:       46.2%
  profit_factor:  1.62
  score (Opt C):  +2.7004

  Trades per ticker:
    AAPL: 6
    AMZN: 14
    AVGO: 6
    BRK.B: 3
    GLD: 4
    GOOGL: 10
    IWM: 2
    JPM: 6
    MSFT: 5
    NVDA: 2
    QLD: 2
    QQQ: 2
    SLV: 4
    SPY: 4
    SQQQ: 2
    TIL: 1
    TLT: 3
    TQQQ: 2


## 4. Configurar y correr optimización (100 trials)

TPESampler con `multivariate=True`, 20 startup trials random, seed=42.
Cada trial corre el pipeline completo sobre los 18 tickers. SwingCache
evita recomputar pasos 1-2 cuando el swing_config se repite entre trials.

In [8]:
N_TRIALS = 50
STUDY_NAME = "vcp_optuna_phase1"
MLFLOW_EXPERIMENT = "autoresearch_vcp_phase1"

In [9]:
import multiprocessing

# Paralelizacion: usar 8 de los 16 cores (dejar margen para sistema + cache contention)
N_JOBS = 1

print(f"CPU cores disponibles: {multiprocessing.cpu_count()}")
print(f"Trials en paralelo: {N_JOBS}")
print(f"Total trials: {N_TRIALS}\n")

cache = SwingCache()
objective = build_objective_function(universe, cache=cache)
study = create_study(study_name=STUDY_NAME, n_startup_trials=20, seed=42)
logger = MLflowOptunaLogger(experiment_name=MLFLOW_EXPERIMENT)

with logger.parent_run(study_name=STUDY_NAME, tags={"n_tickers": str(len(universe)), "n_jobs": str(N_JOBS)}):
    study.optimize(
        objective, 
        n_trials=N_TRIALS, 
        n_jobs=N_JOBS,
        callbacks=[logger.optuna_callback],
        show_progress_bar=True,
    )
    logger.log_study_summary(study, n_tickers=len(universe))

print(f"\nOptimización completada: {len(study.trials)} trials")
print(f"Best score: {study.best_value:+.4f} (trial #{study.best_trial.number})")
print(f"Cache stats: {cache.stats()}")

/home/gdelarosa/proyectos/deteccion-vcp/autoresearch/objective.py:73: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(
[I 2026-05-06 20:04:55,226] A new study created in memory with name: vcp_optuna_phase1


CPU cores disponibles: 16
Trials en paralelo: 1
Total trials: 50



Best trial: 0. Best value: 3.05306:   2%|▏         | 1/50 [00:28<23:21, 28.61s/it]

[I 2026-05-06 20:05:23,902] Trial 0 finished with value: 3.0530573444124185 and parameters: {'atr_length': 15, 'atr_mult': 3.5, 'min_contractions': 3, 'max_contractions': 6, 'lookback_bars': 90, 'tolerance': 0.07500000000000001, 'max_depth_pct': 0.26161672243363987, 'min_total_reduction': 0.8665440364437338, 'compression_threshold': 0.8502787529358021, 'vol_contraction_threshold': 0.891614515559209, 'volume_ratio_threshold': 1.3144091460070617, 'max_gap_days': 40}. Best is trial 0 with value: 3.0530573444124185.


Best trial: 0. Best value: 3.05306:   4%|▍         | 2/50 [05:33<2:32:58, 191.22s/it]

...

[I 2026-05-06 21:21:54,181] Trial 47 finished with value: 1.8769701055556758 and parameters: {'atr_length': 17, 'atr_mult': 3.5, 'min_contractions': 3, 'max_contractions': 6, 'lookback_bars': 130, 'tolerance': 0.05, 'max_depth_pct': 0.3172691108948708, 'min_total_reduction': 0.7687184923349493, 'compression_threshold': 0.9393066171461523, 'vol_contraction_threshold': 0.940926275783861, 'volume_ratio_threshold': 1.7641453489448757, 'max_gap_days': 36}. Best is trial 25 with value: 4.132041111347284.


Best trial: 25. Best value: 4.13204:  98%|█████████▊| 49/50 [1:18:02<00:48, 48.75s/it]

[I 2026-05-06 21:22:57,760] Trial 48 finished with value: 2.071255031163409 and parameters: {'atr_length': 15, 'atr_mult': 3.5, 'min_contractions': 3, 'max_contractions': 7, 'lookback_bars': 130, 'tolerance': 0.1, 'max_depth_pct': 0.3569670311402051, 'min_total_reduction': 0.8475507492840639, 'compression_threshold': 0.8829218314597391, 'vol_contraction_threshold': 0.8906859563565565, 'volume_ratio_threshold': 1.6014173283206738, 'max_gap_days': 40}. Best is trial 25 with value: 4.132041111347284.


Best trial: 25. Best value: 4.13204: 100%|██████████| 50/50 [1:18:57<00:00, 94.74s/it]

[I 2026-05-06 21:23:52,419] Trial 49 finished with value: 2.867067152312463 and parameters: {'atr_length': 13, 'atr_mult': 3.5, 'min_contractions': 3, 'max_contractions': 7, 'lookback_bars': 140, 'tolerance': 0.05, 'max_depth_pct': 0.3140641683570618, 'min_total_reduction': 0.7173674459334882, 'compression_threshold': 0.8696495952820884, 'vol_contraction_threshold': 0.8939627132888232, 'volume_ratio_threshold': 1.420865459428848, 'max_gap_days': 20}. Best is trial 25 with value: 4.132041111347284.

Optimización completada: 50 trials
Best score: +4.1320 (trial #25)
Cache stats: {'hits': 252, 'misses': 648, 'hit_rate': 0.28, 'size': 648}


## 5. Inspeccionar resultados

### 5a. Best trial

In [10]:
best = study.best_trial
print(f"Best trial: #{best.number}")
print(f"  score:         {best.value:+.4f}")
print(f"  n_trades:      {best.user_attrs['n_trades']}")
print(f"  expectancy_r:  {best.user_attrs['expectancy_r']:+.4f}")
print(f"  win_rate:      {best.user_attrs['win_rate']:.1%}")
print(f"  profit_factor: {best.user_attrs['profit_factor']:.2f}")
print(f"\nParams:")
for k, v in sorted(best.params.items()):
    print(f"  {k}: {v}")

Best trial: #25
  score:         +4.1320
  n_trades:      26
  expectancy_r:  +0.8104
  win_rate:      50.0%
  profit_factor: 3.40

Params:
  atr_length: 12
  atr_mult: 3.5
  compression_threshold: 0.8874724117023911
  lookback_bars: 120
  max_contractions: 7
  max_depth_pct: 0.32846215580994775
  max_gap_days: 37
  min_contractions: 3
  min_total_reduction: 0.8602632729686918
  tolerance: 0.05
  vol_contraction_threshold: 0.9263365994864087
  volume_ratio_threshold: 1.5183214679042045


### 5b. Top 10 trials

In [11]:
top10 = top_trials_summary(study, n=10)
display(top10)

,trial_number,score,n_trades,expectancy_r,win_rate,profit_factor,atr_length,atr_mult,min_contractions,max_contractions,lookback_bars,tolerance,max_depth_pct,min_total_reduction,compression_threshold,vol_contraction_threshold,volume_ratio_threshold,max_gap_days
0,25,4.1320,26,0.8104,0.5000,3.4008,12,3.5000,3,7,120,0.0500,0.3285,0.8603,0.8875,0.9263,1.5183,37
1,27,3.4382,29,0.6385,0.4483,2.6930,12,3.5000,3,7,130,0.0750,0.3389,0.8030,0.9088,0.9103,1.6244,38
2,39,3.3559,12,0.9688,0.5833,4.4905,16,3.2500,3,7,80,0.0750,0.2552,0.8449,0.8966,0.8950,1.3431,40
3,6,3.3287,43,0.5076,0.4419,2.5648,10,3.2500,3,7,130,0.0500,0.3217,0.6790,0.9158,0.8747,1.5316,21
4,43,3.3096,8,1.3082,0.6250,9.2185,13,3.5000,3,7,80,0.0500,0.2784,0.8423,0.9224,0.8417,1.3783,36
5,21,3.2885,17,0.7976,0.5294,5.3068,15,3.2500,3,7,90,0.0500,0.2931,0.8559,0.8903,0.9451,1.3912,40
6,38,3.2109,41,0.5015,0.5122,2.2702,16,2.5000,3,7,80,0.0500,0.3184,0.7550,0.8614,0.9284,1.3249,39
7,20,3.1527,16,0.7882,0.5625,4.5021,15,3.0000,3,6,80,0.1000,0.3112,0.8431,0.8670,0.9178,1.4660,40
8,31,3.0658,30,0.5597,0.4667,2.4422,14,3.2500,3,7,110,0.0500,0.3132,0.8501,0.9179,0.9410,1.5552,40
9,0,3.0531,16,0.7633,0.4375,3.8732,15,3.5000,3,6,90,0.0750,0.2616,0.8665,0.8503,0.8916,1.3144,40


### 5c. Importancia de parámetros (fANOVA)

In [12]:
imp = param_importance(study)
if imp is not None:
    display(imp)
else:
    print("param_importance no pudo calcularse (pocos trials o error de fANOVA)")

,param,importance
0,vol_contraction_threshold,0.2082
1,max_gap_days,0.1732
2,volume_ratio_threshold,0.1576
3,min_total_reduction,0.1442
4,max_depth_pct,0.1333
5,atr_mult,0.0527
6,compression_threshold,0.0484
7,lookback_bars,0.0240
8,min_contractions,0.0202
9,atr_length,0.0146


### 5d. Visualizaciones de Optuna

In [13]:
try:
    from optuna.visualization import (
        plot_optimization_history,
        plot_parallel_coordinate,
        plot_param_importances,
        plot_slice,
    )
    HAS_PLOTLY = True
except ImportError:
    HAS_PLOTLY = False
    print("plotly no instalado. Correr: pip install plotly")

In [14]:
if HAS_PLOTLY:
    fig = plot_optimization_history(study)
    fig.update_layout(title="Optimization History", height=400)
    fig.show()

In [15]:
if HAS_PLOTLY:
    fig = plot_parallel_coordinate(
        study,
        params=[
            "atr_mult", "tolerance", "max_depth_pct",
            "compression_threshold", "vol_contraction_threshold",
            "volume_ratio_threshold",
        ],
    )
    fig.update_layout(title="Parallel Coordinate (top params)", height=500)
    fig.show()

In [16]:
if HAS_PLOTLY:
    try:
        fig = plot_param_importances(study)
        fig.update_layout(title="Parameter Importances (fANOVA)", height=400)
        fig.show()
    except Exception as e:
        print(f"plot_param_importances falló: {e}")

In [17]:
if HAS_PLOTLY:
    fig = plot_slice(
        study,
        params=["atr_mult", "tolerance", "compression_threshold", "volume_ratio_threshold"],
    )
    fig.update_layout(title="Slice Plot (key params vs score)", height=400)
    fig.show()

## 6. Re-ejecutar best trial para validar reproducibilidad

In [18]:
best_params = reconstruct_pipeline_params(study.best_trial.params)
rerun_result = run_backtest_for_params(universe, best_params)
rerun_metrics = rerun_result["metrics"]
rerun_score = compute_objective_score(rerun_metrics)

original_score = study.best_value
score_diff = abs(rerun_score - original_score)

print(f"Score original:    {original_score:+.6f}")
print(f"Score re-run:      {rerun_score:+.6f}")
print(f"Diferencia:        {score_diff:.2e}")

if score_diff > 1e-6:
    print("\n⚠ WARNING: scores difieren. Posible floating point o non-determinism.")
    print(f"  Original n_trades: {study.best_trial.user_attrs['n_trades']}")
    print(f"  Re-run n_trades:   {rerun_metrics['n_trades']}")
else:
    print("\nReproducibilidad OK: scores idénticos.")

Score original:    +4.132041
Score re-run:      +4.132041
Diferencia:        0.00e+00

Reproducibilidad OK: scores idénticos.


In [19]:
best_trades_df = trades_to_dataframe(rerun_result["all_trades"])
print(f"Trades del best trial: {len(best_trades_df)}")
display(best_trades_df)

Trades del best trial: 26


,trade_num,ticker,entry_date,exit_date,exit_reason,duration_days,entry_price,exit_price,pnl_pct,r_multiple,max_r,n_contractions,stop_method,stop_distance_pct
0,1,AAPL,2016-07-27,2016-09-08,distribution,43,25.7375,26.3800,0.0250,0.3936,1.0000,3,pattern,0.0634
1,2,AAPL,2016-09-08,2016-09-09,distribution,1,26.3800,25.7825,-0.0226,-0.3236,0.0000,3,fixed_pct,0.0700
2,3,AAPL,2024-12-20,2025-01-13,stop_loss,24,254.4900,234.4000,-0.0789,-1.1277,0.2543,3,fixed_pct,0.0700
3,4,AMZN,2017-04-04,2017-06-09,distribution,66,45.3415,48.9155,0.0788,1.1261,1.6464,3,fixed_pct,0.0700
4,5,AMZN,2017-10-27,2017-12-04,distribution,38,55.0475,56.6975,0.0300,0.4282,1.2311,3,fixed_pct,0.0700
5,6,AVGO,2016-06-03,2016-06-24,stop_loss,21,16.2560,14.8720,-0.0851,-1.2163,0.2004,3,fixed_pct,0.0700
6,7,AVGO,2024-12-13,2025-01-27,stop_loss,45,224.8000,202.1300,-0.1008,-1.4406,1.6014,3,fixed_pct,0.0700
7,8,AVGO,2025-01-27,2025-01-28,distribution,1,202.1300,207.3600,0.0259,0.3696,0.3696,3,fixed_pct,0.0700
8,9,BRK.B,2017-09-05,2018-02-05,distribution,153,176.9800,196.8000,0.1120,1.9168,3.8946,3,pattern,0.0584
9,10,GLD,2025-08-29,2026-01-30,trailing_stop,154,318.0700,444.9500,0.3989,7.4134,10.3903,3,pattern,0.0538


## 7. Tabla comparativa: baseline vs best Optuna

In [20]:
comparison = pd.DataFrame({
    "Baseline (original)": {
        "score": baseline_score,
        "n_trades": baseline_metrics["n_trades"],
        "expectancy_r": baseline_metrics["expectancy_r"],
        "win_rate": baseline_metrics["win_rate"],
        "profit_factor": baseline_metrics["profit_factor"],
        "avg_winner_r": baseline_metrics["avg_winner_r"],
        "avg_loser_r": baseline_metrics["avg_loser_r"],
    },
    "Best Optuna": {
        "score": rerun_score,
        "n_trades": rerun_metrics["n_trades"],
        "expectancy_r": rerun_metrics["expectancy_r"],
        "win_rate": rerun_metrics["win_rate"],
        "profit_factor": rerun_metrics["profit_factor"],
        "avg_winner_r": rerun_metrics["avg_winner_r"],
        "avg_loser_r": rerun_metrics["avg_loser_r"],
    },
}).T

display(comparison)

,score,n_trades,expectancy_r,win_rate,profit_factor,avg_winner_r,avg_loser_r
Baseline (original),2.7004,78.0000,0.3058,0.4615,1.6237,1.7247,-0.9105
Best Optuna,4.1320,26.0000,0.8104,0.5000,3.4008,2.2958,-0.6751


In [21]:
print("=== Parámetros del baseline vs best Optuna ===")
print(f"{'Parámetro':<30} {'Baseline':>12} {'Optuna':>12}")
print("-" * 56)

baseline_flat = {
    "atr_length": 14, "atr_mult": 2.0,
    "min_contractions": 2, "max_contractions": 6,
    "lookback_bars": 126, "tolerance": 0.10,
    "max_depth_pct": 0.35, "min_total_reduction": 0.80,
    "compression_threshold": 0.85, "vol_contraction_threshold": 0.85,
    "volume_ratio_threshold": 1.5, "max_gap_days": 30,
}
optuna_flat = study.best_trial.params

for k in baseline_flat:
    bval = baseline_flat[k]
    oval = optuna_flat.get(k, "N/A")
    if isinstance(bval, float):
        print(f"{k:<30} {bval:>12.4f} {oval:>12.4f}")
    else:
        print(f"{k:<30} {bval:>12} {oval:>12}")

=== Parámetros del baseline vs best Optuna ===
Parámetro                          Baseline       Optuna
--------------------------------------------------------
atr_length                               14           12
atr_mult                             2.0000       3.5000
min_contractions                          2            3
max_contractions                          6            7
lookback_bars                           126          120
tolerance                            0.1000       0.0500
max_depth_pct                        0.3500       0.3285
min_total_reduction                  0.8000       0.8603
compression_threshold                0.8500       0.8875
vol_contraction_threshold            0.8500       0.9263
volume_ratio_threshold               1.5000       1.5183
max_gap_days                             30           37


## 8. Distribución de trades por ticker (best trial)

In [22]:
ticker_comparison = []
for ticker in sorted(universe.keys()):
    bl_n = baseline_result["per_ticker"].get(ticker, {}).get("n_trades", 0)
    opt_n = rerun_result["per_ticker"].get(ticker, {}).get("n_trades", 0)
    ticker_comparison.append({"ticker": ticker, "baseline_trades": bl_n, "optuna_trades": opt_n})

tc_df = pd.DataFrame(ticker_comparison).set_index("ticker")
tc_df["diff"] = tc_df["optuna_trades"] - tc_df["baseline_trades"]
display(tc_df)
print(f"\nBaseline total: {tc_df['baseline_trades'].sum()}")
print(f"Optuna total:   {tc_df['optuna_trades'].sum()}")

,baseline_trades,optuna_trades,diff
ticker,,,
AAPL,6,3,-3
AMZN,14,2,-12
AVGO,6,3,-3
BRK.B,3,1,-2
GLD,4,1,-3
GOOGL,10,1,-9
IWM,2,1,-1
JPM,6,2,-4
MSFT,5,0,-5



Baseline total: 78
Optuna total:   26


## 9. Todos los trials (DataFrame completo)

In [23]:
all_trials_df = study_to_dataframe(study)
print(f"Total trials: {len(all_trials_df)}")
print(f"Trials con score > 0: {(all_trials_df['score'] > 0).sum()}")
print(f"Trials con n_trades == 0: {(all_trials_df['n_trades'] == 0).sum()}")
print(f"Trials con n_trades >= 10: {(all_trials_df['n_trades'] >= 10).sum()}")
display(all_trials_df.head(20))

Total trials: 50
Trials con score > 0: 48
Trials con n_trades == 0: 0
Trials con n_trades >= 10: 44


,trial_number,score,state,atr_length,atr_mult,min_contractions,max_contractions,lookback_bars,tolerance,max_depth_pct,...,vol_contraction_threshold,volume_ratio_threshold,max_gap_days,n_trades,expectancy_r,win_rate,profit_factor,avg_winner_r,avg_loser_r,trades_per_ticker
0,25,4.1320,COMPLETE,12,3.5000,3,7,120,0.0500,0.3285,...,0.9263,1.5183,37,26,0.8104,0.5000,3.4008,2.2958,-0.6751,"{'AAPL': 3, 'AMZN': 2, 'AVGO': 3, 'BRK.B': 1, ..."
1,27,3.4382,COMPLETE,12,3.5000,3,7,130,0.0750,0.3389,...,0.9103,1.6244,38,29,0.6385,0.4483,2.6930,2.2655,-0.6835,"{'AAPL': 3, 'AMZN': 3, 'AVGO': 3, 'BRK.B': 1, ..."
2,39,3.3559,COMPLETE,16,3.2500,3,7,80,0.0750,0.2552,...,0.8950,1.3431,40,12,0.9688,0.5833,4.4905,2.1365,-0.6661,"{'AAPL': 1, 'AMZN': 2, 'AVGO': 1, 'GLD': 1, 'N..."
3,6,3.3287,COMPLETE,10,3.2500,3,7,130,0.0500,0.3217,...,0.8747,1.5316,21,43,0.5076,0.4419,2.5648,1.8830,-0.5812,"{'AAPL': 3, 'AMZN': 3, 'AVGO': 5, 'BRK.B': 5, ..."
4,43,3.3096,COMPLETE,13,3.5000,3,7,80,0.0500,0.2784,...,0.8417,1.3783,36,8,1.3082,0.6250,9.2185,2.3479,-0.4245,"{'AAPL': 1, 'AMZN': 2, 'GLD': 2, 'TIL': 1, 'TL..."
5,21,3.2885,COMPLETE,15,3.2500,3,7,90,0.0500,0.2931,...,0.9451,1.3912,40,17,0.7976,0.5294,5.3068,1.8563,-0.3935,"{'AAPL': 2, 'AMZN': 2, 'AVGO': 1, 'BRK.B': 1, ..."
6,38,3.2109,COMPLETE,16,2.5000,3,7,80,0.0500,0.3184,...,0.9284,1.3249,39,41,0.5015,0.5122,2.2702,1.7498,-0.8093,"{'AAPL': 2, 'AMZN': 10, 'GLD': 2, 'GOOGL': 3, ..."
7,20,3.1527,COMPLETE,15,3.0000,3,6,80,0.1000,0.3112,...,0.9178,1.4660,40,16,0.7882,0.5625,4.5021,1.8013,-0.5144,"{'AAPL': 1, 'AMZN': 3, 'AVGO': 1, 'GLD': 1, 'I..."
8,31,3.0658,COMPLETE,14,3.2500,3,7,110,0.0500,0.3132,...,0.9410,1.5552,40,30,0.5597,0.4667,2.4422,2.0311,-0.7277,"{'AAPL': 4, 'AMZN': 4, 'AVGO': 3, 'BRK.B': 1, ..."
9,0,3.0531,COMPLETE,15,3.5000,3,6,90,0.0750,0.2616,...,0.8916,1.3144,40,16,0.7633,0.4375,3.8732,2.3518,-0.4723,"{'AAPL': 2, 'AMZN': 1, 'BRK.B': 3, 'GLD': 2, '..."


## 10. Conclusiones y próximos pasos

### Resultados
- **Baseline score** vs **Best Optuna score**: ver tabla comparativa arriba.
- Los parámetros que más impactan el score se ven en la tabla de fANOVA importance.
- La distribución de trades por ticker muestra si Optuna concentra señales en pocos tickers o diversifica.

### Limitaciones de esta Fase 1
- **In-sample optimization**: estamos optimizando y evaluando sobre el mismo periodo. No hay validación out-of-sample.
- **Sin walk-forward**: no probamos si los parámetros generalizan a periodos futuros.
- **Solo `method=tolerance`**: no exploramos `robust_trend` ni otros métodos de compresión.
- **50 trials**: con 12 parámetros, TPE necesita más trials para convergir bien (~200-500).

### Próximos pasos (Fase 2)
1. **Walk-forward validation**: dividir el periodo en ventanas train/test y optimizar rolling.
2. **Out-of-sample test**: correr best params sobre los tickers excluidos (COIN, HOOD, PLTR, SOFI).
3. **Más trials**: escalar a 200+ trials para explorar mejor el espacio.
4. **Explorar métodos**: agregar `robust_trend` y `ratio_normalized` al search space.
5. **Multi-objective**: optimizar expectancy y n_trades como objetivos separados (Pareto front).

### MLflow
Para explorar los resultados en detalle:
```bash
cd /home/gdelarosa/proyectos/deteccion-vcp && mlflow ui --backend-store-uri mlruns/
```

In [26]:
from models.configs import ATRZigZagConfig

# Workaround temporal: convertir swing_config de dict a ATRZigZagConfig
# (bug en results.py - reconstruct_pipeline_params devuelve dict pero el cache espera objeto)
best_params = reconstruct_pipeline_params(study.best_trial.params)
if isinstance(best_params["swing_config"], dict):
    best_params["swing_config"] = ATRZigZagConfig(**best_params["swing_config"])

# Lo mismo para baseline_params (es un dict literal, también necesita convertirse)
if isinstance(baseline_params["swing_config"], dict):
    baseline_params["swing_config"] = ATRZigZagConfig(**baseline_params["swing_config"])

print("Re-corriendo baseline y best Optuna (con cache deberia ser rapido)...")
base_result = run_backtest_for_params(universe, baseline_params, cache=cache)
best_result = run_backtest_for_params(universe, best_params, cache=cache)

Re-corriendo baseline y best Optuna (con cache deberia ser rapido)...
